# PCT Tuning and Eval

Uses checkpoints to fine-tune PCT models with the other dataset (full, partial) that they were originally trained on.

## Prep

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!git clone --recursive --single-branch --branch PCT-retrain https://github.com/DavidClaszen/pointcloud-bench /content/pointcloud-bench
%cd /content/pointcloud-bench
%pip install -r envs/pct/requirements.txt

Cloning into '/content/pointcloud-bench'...
remote: Enumerating objects: 344, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 344 (delta 63), reused 74 (delta 52), pack-reused 242 (from 1)
Receiving objects: 100% (344/344), 39.52 MiB | 26.76 MiB/s, done.
Resolving deltas: 100% (146/146), done.
Submodule 'repos/PAPNet' (https://github.com/DavidClaszen/PAPNet.git) registered for path 'repos/PAPNet'
Submodule 'repos/Point-Transformers' (https://github.com/DavidClaszen/Point-Transformers.git) registered for path 'repos/Point-Transformers'
Cloning into '/content/pointcloud-bench/repos/PAPNet'...
remote: Enumerating objects: 133, done.        
remote: Counting objects: 100% (133/133), done.        
remote: Compressing objects: 100% (105/105), done.        
remote: Total 133 (delta 55), reused 85 (delta 27), pack-reused 0 (from 0)        
Receiving objects: 100% (133/133), 9.66 MiB | 12.89 MiB/s, done.
Resolving deltas: 100%

In [3]:
# Paths, folders
import os

REPO_PATH = '/content/pointcloud-bench'
DRIVE_PATH = '/content/drive/MyDrive/pointcloud-bench'
results_dir = os.path.join(DRIVE_PATH, 'results')
os.makedirs(results_dir, exist_ok=True)

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, confusion_matrix
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
from IPython.display import display, HTML
from tqdm import tqdm

In [5]:
# Copy select files from Google Drive to the repo dataset folder
!rsync -avP {DRIVE_PATH}/datasets/fullmodelnet40.tar.gz {REPO_PATH}/datasets
!rsync -avP {DRIVE_PATH}/datasets/partialmodelnet40.tar.gz {REPO_PATH}/datasets
!rsync -avP {DRIVE_PATH}/datasets/pm40_extreme_partiality.tar.gz {REPO_PATH}/datasets

# Extract zip files
%cd {REPO_PATH}
!tar -xvzf datasets/fullmodelnet40.tar.gz -C datasets
!tar -xvzf datasets/partialmodelnet40.tar.gz -C datasets
!tar -xvzf datasets/pm40_extreme_partiality.tar.gz -C datasets

sending incremental file list
fullmodelnet40.tar.gz
  1,833,210,387 100%   42.91MB/s    0:00:40 (xfr#1, to-chk=0/1)

sent 1,833,658,058 bytes  received 35 bytes  43,144,896.31 bytes/sec
total size is 1,833,210,387  speedup is 1.00
sending incremental file list
partialmodelnet40.tar.gz
  2,162,130,396 100%   42.92MB/s    0:00:48 (xfr#1, to-chk=0/1)

sent 2,162,658,366 bytes  received 35 bytes  43,690,068.71 bytes/sec
total size is 2,162,130,396  speedup is 1.00
sending incremental file list
pm40_extreme_partiality.tar.gz
  8,334,013,260 100%   45.34MB/s    0:02:55 (xfr#1, to-chk=0/1)

sent 8,336,048,044 bytes  received 35 bytes  46,963,651.15 bytes/sec
total size is 8,334,013,260  speedup is 1.00
/content/pointcloud-bench
fullmodelnet40/
fullmodelnet40/test_filenames.txt
fullmodelnet40/test_gt_rot.npy
fullmodelnet40/test_gt_tra.npy
fullmodelnet40/test_labels.npy
fullmodelnet40/test_points.npy
fullmodelnet40/train_filenames.txt
fullmodelnet40/train_gt_rot.npy
fullmodelnet40/train_gt_tra.

In [ ]:
# Original data used by Point-Transformers, from ShapeNet
# Kept for archival reasons; uncomment if required later
# ZIP = '/content/drive/MyDrive/pointcloud-bench/datasets/modelnet40_normal_resampled.zip'
# DEST = '/content/pointcloud-bench/datasets/modelnet40_pct'
# !unzip -q -n '{ZIP}' -d '{DEST}'
# !find '{DEST}' -type f | wc -l

## Checkpoints

You can get the trained model checkpoints from [this folder](https://drive.google.com/drive/folders/13TIEUMSkxi-MxY_Y-kTkAsEcRP5OgAR3?usp=sharing).

Copy those files into your own Google Drive and, if necessary, change the first path in the code below. Contents:

- Main models:
    - pct_p50.pth:      PCT trained on partial50 from PAPNet
    - pct_pfull.pth:    PCT trained on full version of PAPNet
- Archived:
    - pct_pct.pth:      PCT trained on the original dataset used for PCT
    - pct_pct_p50:      PCT trained on the original dataset used for PCT, tuned on partial50 from PAPNet
    - pct_p50_pct.pth:  PCT trained on partial 50 from PAPNet, tuned on original dataset used for PCT

In [6]:
!rsync -avP {DRIVE_PATH}/checkpoints/. {REPO_PATH}/checkpoints/

sending incremental file list
./
pct_p50.pth
     34,642,219 100%   17.48MB/s    0:00:01 (xfr#1, to-chk=6/8)
pct_p50_pct.pth
     34,642,524 100%   11.05MB/s    0:00:02 (xfr#2, to-chk=5/8)
pct_p50_pfull.pth
     34,641,253 100%   10.57MB/s    0:00:03 (xfr#3, to-chk=4/8)
pct_pct.pth
     34,642,219 100%   10.16MB/s    0:00:03 (xfr#4, to-chk=3/8)
pct_pct_p50.pth
     34,641,609 100%    9.62MB/s    0:00:03 (xfr#5, to-chk=2/8)
pct_pfull.pth
     34,642,219 100%    8.42MB/s    0:00:03 (xfr#6, to-chk=1/8)
pct_pfull_p50.pth
     34,642,379 100%   10.91MB/s    0:00:03 (xfr#7, to-chk=0/8)

sent 242,554,107 bytes  received 152 bytes  9,511,931.73 bytes/sec
total size is 242,494,422  speedup is 1.00


## Tuning


In [7]:
%cd /content/pointcloud-bench/repos/Point-Transformers
!python train_cls.py --help

/content/pointcloud-bench/repos/Point-Transformers
train_cls is powered by Hydra.

== Configuration groups ==
Compose your configuration from those groups (group=option)

model: Hengshuang, Menghao, Nico


== Config ==
Override anything in the config (foo.bar=value)

model:
  name: Menghao
batch_size: 16
epoch: 200
learning_rate: 0.001
gpu: 0
num_point: 1024
optimizer: Adam
weight_decay: 0.0001
normal: true
use_papnet_loader: false
workers: 2
step_size: 50
data_path: ../../datasets/modelnet40_normal_resampled/
checkpoint_path: best_model.pth
partiality: ''


Powered by Hydra (https://hydra.cc)
Use --hydra-help to view Hydra specific help




In [ ]:
# Tune PCT model trained on full data further on partial (partialmodelnet40, and pct_pfull.pth)
!python train_cls.py model=Menghao use_papnet_loader=True batch_size=256 learning_rate=0.0005 epoch=85 workers=4 data_path=../../datasets/partialmodelnet40/ checkpoint_path=../../checkpoints/pct_pfull.pth step_size=5

In [13]:
# Tune PCT model trained on partial data further on full (fullmodelnet40, and pct_p50.pth)
!python train_cls.py model=Menghao use_papnet_loader=True batch_size=256 learning_rate=0.0005 epoch=85 workers=4 data_path=../../datasets/fullmodelnet40/ checkpoint_path=../../checkpoints/pct_p50.pth step_size=5

model:
  name: Menghao
batch_size: 256
epoch: 85
learning_rate: 0.0005
gpu: 0
num_point: 1024
optimizer: Adam
weight_decay: 0.0001
normal: true
use_papnet_loader: true
workers: 4
step_size: 5
data_path: ../../datasets/fullmodelnet40/
checkpoint_path: ../../checkpoints/pct_p50.pth
partiality: ''

[2025-12-01 05:12:29,961][__main__][INFO] - Load dataset ...
The size of train data is 98430
The size of test data is 2468
[2025-12-01 05:12:30,963][__main__][INFO] - Use pretrain model
[2025-12-01 05:12:31,959][__main__][INFO] - Start training...
[2025-12-01 05:12:31,959][__main__][INFO] - Epoch 1 (39/85):
100% 385/385 [02:34<00:00,  2.50it/s]
[2025-12-01 05:15:06,257][__main__][INFO] - Train Instance Accuracy: 0.701321
100% 10/10 [00:03<00:00,  3.16it/s]
[2025-12-01 05:15:09,501][__main__][INFO] - Test Instance Accuracy: 0.734613, Class Accuracy: 0.699615
[2025-12-01 05:15:09,501][__main__][INFO] - Best Instance Accuracy: 0.734613, Class Accuracy: 0.699615
[2025-12-01 05:15:09,501][__main__][

In [15]:
# Zip Point-Transformers logs
!zip -r results.zip ./log/cls/Menghao/

updating: log/cls/Menghao/ (stored 0%)
updating: log/cls/Menghao/model.py (deflated 76%)
updating: log/cls/Menghao/train_cls.log (deflated 88%)
updating: log/cls/Menghao/.hydra/ (stored 0%)
updating: log/cls/Menghao/.hydra/config.yaml (deflated 31%)
updating: log/cls/Menghao/.hydra/overrides.yaml (deflated 27%)
updating: log/cls/Menghao/.hydra/hydra.yaml (deflated 66%)


## Eval

In [19]:
# Function for inference

from test_cls import get_predictions

%cd {REPO_PATH}/repos/Point-Transformers/
config_dir = f'{REPO_PATH}/repos/Point-Transformers/config'


def test_pct(
    data_folder: str = 'partialmodelnet40',
    checkpoint: str = 'pct_p50',
    config_dir: str = config_dir,
    partiality: str = ''
) -> tuple[np.array, np.array]:
    """Gets predictions from pretrained PCT model.

    Args:
        data_folder (str, optional): Dataset folder to test on.
            Defaults to 'partialmodelnet40'.
        checkpoint (str, optional): Checkpoint name; omit extension.
            Defaults to 'pct_p50'.
        config_dir (str, optional): Where the model configs reside.
            Only necessary to run, but not used. Defaults to config_dir.
        partiality (str, optional): Used only for extreme partiality
            datasets with non-standard filenames. Use '_10', '_20',
            '_30', '_40'.

    Returns:
        tuple(np.array, np.array): Tuple of predictions and base truths
    """
    if data_folder in ['partialmodelnet40', 'pm40_partiality_level', 'fullmodelnet40']:
        papnet_loader = 'true'
    else: papnet_loader = 'false'

    with initialize_config_dir(config_dir=config_dir, version_base='1.2'):
        cfg = compose(
            config_name='cls',
            overrides=[
                f'data_path=../../datasets/{data_folder}/',
                f'checkpoint_path=../../checkpoints/{checkpoint}.pth',
                f'use_papnet_loader={papnet_loader}',
                f'partiality={partiality}'
            ],
        )
    OmegaConf.set_struct(cfg, False)
    y_true, y_pred = get_predictions(cfg)
    accuracy = accuracy_score(y_true, y_pred)
    print(f'Accuracy: {accuracy:.4f} for dataset {data_folder} and model {checkpoint}')
    return (y_true, y_pred)


results_dir = os.path.join(DRIVE_PATH, 'results')
os.makedirs(results_dir, exist_ok=True)

/content/pointcloud-bench/repos/Point-Transformers


In [20]:
# Train PCT on fullmodelnet40, test on partialmodelnet40 (tuned on partialmodelnet40)
y_true, y_pred = test_pct(
    data_folder='partialmodelnet40',
    checkpoint='pct_pfull_p50'
  )
result_df = pd.DataFrame({
    'y_true': y_true,
    'full_p50_tunep50)': y_pred
})

# Train PCT on fullmodelnet40, test on fullmodelnet40 (tuned on partialmodelnet40)
y_true, y_pred = test_pct(
    data_folder='fullmodelnet40',
    checkpoint='pct_pfull_p50'
  )
result_df['full_full_tunep50'] = y_pred

# Train PCT on partialmodelnet40, test on partialmodelnet40 (tuned on fullmodelnet40)
y_true, y_pred = test_pct(
    data_folder='partialmodelnet40',
    checkpoint='pct_p50_pfull'
  )

result_df['p50_p50_tunefull'] = y_pred

# Train PCT on partialmodelnet40, test on fullmodelnet40 (tuned on fullmodelnet40)
y_true, y_pred = test_pct(
    data_folder='fullmodelnet40',
    checkpoint='pct_p50_pfull'
  )

result_df['p50_pfull_tunefull'] = y_pred

The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 53 epochs
# iterations (batches): 155
Average inference time per batch: 171.561 ms
Average inference time per sample: 10.774680 ms
Accuracy: 0.8092 for dataset partialmodelnet40 and model pct_pfull_p50
The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 53 epochs
# iterations (batches): 155
Average inference time per batch: 133.533 ms
Average inference time per sample: 8.386398 ms
Accuracy: 0.2318 for dataset fullmodelnet40 and model pct_pfull_p50
The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 73 epochs
# iterations (batches): 155
Average inference time per batch: 130.116 ms
Average inference time per sample: 8.171774 ms
Accuracy: 0.2358 for dataset partialmodelnet40 and model pct_p50_pfull
The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 73 epochs
# iterations (batches): 155
Average inference time pe

In [21]:
# Extreme partialities, full model, tuned partial

y_true, y_pred = test_pct(data_folder='pm40_partiality_level', checkpoint='pct_pfull_p50', partiality='_40')
result_df['full_partial40_tunep50'] = y_pred
y_true, y_pred = test_pct(data_folder='pm40_partiality_level', checkpoint='pct_pfull_p50', partiality='_30')
result_df['full_partial30_tunep50'] = y_pred
y_true, y_pred = test_pct(data_folder='pm40_partiality_level', checkpoint='pct_pfull_p50', partiality='_20')
result_df['full_partial20_tunep50'] = y_pred
y_true, y_pred = test_pct(data_folder='pm40_partiality_level', checkpoint='pct_pfull_p50', partiality='_10')
result_df['full_partial10_tunep50'] = y_pred

The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 53 epochs
# iterations (batches): 155
Average inference time per batch: 146.110 ms
Average inference time per sample: 9.176287 ms
Accuracy: 0.6556 for dataset pm40_partiality_level and model pct_pfull_p50
The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 53 epochs
# iterations (batches): 155
Average inference time per batch: 126.301 ms
Average inference time per sample: 7.932182 ms
Accuracy: 0.4254 for dataset pm40_partiality_level and model pct_pfull_p50
The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 53 epochs
# iterations (batches): 155
Average inference time per batch: 163.631 ms
Average inference time per sample: 10.276665 ms
Accuracy: 0.2605 for dataset pm40_partiality_level and model pct_pfull_p50
The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 53 epochs
# iterations (batches): 155
Average in

In [22]:
# Extreme partialities, partial model, tuned full
y_true, y_pred = test_pct(data_folder='pm40_partiality_level', checkpoint='pct_p50_pfull', partiality='_40')
result_df['partial50_partial40_tunefull'] = y_pred
y_true, y_pred = test_pct(data_folder='pm40_partiality_level', checkpoint='pct_p50_pfull', partiality='_30')
result_df['partial50_partial30_tunefull'] = y_pred
y_true, y_pred = test_pct(data_folder='pm40_partiality_level', checkpoint='pct_p50_pfull', partiality='_20')
result_df['partial50_partial20_tunefull'] = y_pred
y_true, y_pred = test_pct(data_folder='pm40_partiality_level', checkpoint='pct_p50_pfull', partiality='_10')
result_df['partial50_partial10_tunefull'] = y_pred


The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 73 epochs
# iterations (batches): 155
Average inference time per batch: 156.403 ms
Average inference time per sample: 9.822688 ms
Accuracy: 0.1637 for dataset pm40_partiality_level and model pct_p50_pfull
The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 73 epochs
# iterations (batches): 155
Average inference time per batch: 174.128 ms
Average inference time per sample: 10.935924 ms
Accuracy: 0.1122 for dataset pm40_partiality_level and model pct_p50_pfull
The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 73 epochs
# iterations (batches): 155
Average inference time per batch: 133.515 ms
Average inference time per sample: 8.385255 ms
Accuracy: 0.0685 for dataset pm40_partiality_level and model pct_p50_pfull
The size of test data is 2468
Total number of parameters: 2876328
Model loaded with 73 epochs
# iterations (batches): 155
Average in

In [31]:
# Store results
results_dir = os.path.join(DRIVE_PATH, 'results')
os.makedirs(results_dir, exist_ok=True)
result_df.to_csv(os.path.join(results_dir, 'pct_results_tuning.csv'), index=False, sep=';')

# Table results
accuracies = []
columns = result_df.columns[1:]

for col in columns:
  accuracy = accuracy_score(y_true, result_df[col])
  accuracies.append(accuracy * 100)

accuracy_df = pd.DataFrame({
    'Trained On': columns,
    'Accuracy (%)': accuracies
})

accuracy_df[['Trained On', 'Tested On', 'Tuned On']] = accuracy_df['Trained On'].str.split('_', expand=True)
accuracy_df['Accuracy (%)'] = accuracy_df['Accuracy (%)'].map("{:.2f}".format)
accuracy_df = accuracy_df.sort_values(by=['Trained On', 'Tested On'])[['Trained On', 'Tested On', 'Tuned On', 'Accuracy (%)']].reset_index(drop=True)
accuracy_df['Tuned On'] = accuracy_df['Tuned On'].str.replace('tune|\\)', '', regex=True)
display(HTML(accuracy_df.to_html()))

,Trained On,Tested On,Tuned On,Accuracy (%)
0,full,full,p50,23.18
1,full,p50,p50,80.92
2,full,partial10,p50,11.26
3,full,partial20,p50,26.05
4,full,partial30,p50,42.54
5,full,partial40,p50,65.56
6,p50,p50,full,23.58
7,p50,pfull,full,88.57
8,partial50,partial10,full,3.77
9,partial50,partial20,full,6.85
